In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display
from sklearn.preprocessing import MinMaxScaler
from semopy import Model, calc_stats, semplot, report
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import seaborn as sns
import pingouin as pg

In [ ]:
df = pd.read_spss('C:\\data\\education\\한국아동청소년행복지수조사\\kor_data_20210010.sav')
df.head(3)

In [ ]:
df['q4812'].value_counts()

In [ ]:
# df = df[df['school'] == '고등학생']
# df.head(3)

In [ ]:
len(df)

In [ ]:
def null0(df):
    for col in df.columns:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(0)
    return df

In [ ]:
def encoding(df, columns, category_map): 
    df[columns] = df[columns].replace(category_map)
    return df

category4_map = {
    '전혀 그렇지 않다': 0,
    '그렇지 않다': 1,
    '그렇다': 2,
    '매우 그렇다': 3
}
category5_map = {
    '전혀 그렇지 않다': 0,
    '별로 그렇지 않다': 1,
    '보통이다' : 2,
    '조금 그렇다': 3,
    '매우 그렇다': 4
}
reverse_category5_map = {
    '전혀 그렇지 않다': 4,
    '별로 그렇지 않다': 3,
    '보통이다' : 2,
    '조금 그렇다': 1,
    '매우 그렇다': 0
}
how_serious_map = {
    '전혀 심각하지 않다' : 0,
    '별로 심각하지 않다' : 1,
    '보통이다' : 2,
    '약간 심각한 편이다' : 3,
    '매우 심각하다' : 4
}
how_often_map = {
'경험이 전혀 없거나 거의 없었다' : 2,
'1년에 몇 번 있었다' : 1,
'한 달에 몇 번 있었다' : 0,
'일주일에 한번 이상 있었다' : 0 
}
happy_map = {
'매우 행복하다' : 4,
'대체로 행복한 편이다' : 3,
'모르겠다' : 2,
'별로 행복하지 않은 편이다' : 1,
'전혀 행복하지 않다' : 0
}
how_happy_map = {
'매우 행복하다' : 4,
'약간 행복하다' : 3,
'보통이다' : 2,
'약간 불행하다' : 1,
'매우 불행하다' : 0
}
yes_or_no_map = {
'예' : 1,
'아니오' : 0
}
is_or_not_map = {
    '있다' : 1,
    '없다' : 0
}
is_or_not_map2 = {
    '있다' : 2,
    '없다' : 1
}
how_much_is_map = {
    '전혀 없다' : 0,
    '1~2번 있다' : 1,
    '3~4번 있다' : 2,
    '5번 이상 있다' : 2
}
how_high_map = {
    '상의 상' : 2,
    '상의 하': 2,
    '중의 상': 1,
    '중의 하': 1,
    '하의 상': 0, 
    '하의 하': 0
}

In [ ]:
def time_min(df, hour_column, minute_column):
    # 시간과 분 열의 결측값을 0으로 채움
    df[hour_column] = df[hour_column].fillna(0)
    df[minute_column] = df[minute_column].fillna(0)
    
    # 데이터 타입을 float로 변환
    df[hour_column] = df[hour_column].astype(float)
    df[minute_column] = df[minute_column].astype(float)

    # 총 분 계산
    df['pri_edu_time'] = df[hour_column] * 60 + df[minute_column]
    return df

In [ ]:
def not_applicable(df, columns):
    for col in columns:
        df[col] = df[col].replace({'해당사항 없음': '보통이다'})
    return df
def applicable_is_the_worst(df, columns):
    for col in columns:
        df[col] = df[col].replace({'해당사항 없음': '전혀 그렇지 않다'})
    return df
def applicable_is_the_best(df, columns):
    for col in columns:
        df[col] = df[col].replace({'해당사항 없음': '매우 그렇다'})
    return df

In [ ]:
def weighted_cov(X, weights):
    average = np.average(X, axis=0, weights=weights)
    X_centered = X - average
    cov_matrix = np.dot((X_centered * weights[:, None]).T, X_centered) / (weights.sum() - 1)
    return cov_matrix

In [ ]:
def calculate_covariance(df, observed_vars):
    data_for_cov = df[observed_vars].apply(pd.to_numeric, errors='coerce')
    weights = df['wt'].to_numpy()
    w_cov = weighted_cov(data_for_cov.to_numpy(), weights)
    return pd.DataFrame(w_cov, index=observed_vars, columns=observed_vars)

In [ ]:
def compute_vif(df, observed_vars):
    X = df[observed_vars].apply(pd.to_numeric, errors='coerce').dropna()
    vif_data = pd.DataFrame()
    vif_data["Feature"] = X.columns
    vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    return vif_data

In [ ]:
def fit_sem_model(model, df, w_cov_df):
    model.fit(df, cov=w_cov_df)
    return model

In [ ]:
def calculate_bmi(df, height, weight):
    df[height] = pd.to_numeric(df[height], errors='coerce')
    df[weight] = pd.to_numeric(df[weight], errors='coerce')
    
    df['bmi'] = df[weight] / ((df[height] / 100) ** 2)
    df['bmi'] = df['bmi'].round(2)
    return df

In [ ]:
# 표준화 계수 계산 함수
def standardize_estimates(df, data):
    std_estimates = []
    for _, row in df.iterrows():
        if row['op'] == '~':  # 회귀 계수
            try:
                x_std = np.std(data[row['rhs']], ddof=1)
                y_std = np.std(data[row['lhs']], ddof=1)
                std_beta = row['Estimate'] * (x_std / y_std)
                std_estimates.append(std_beta)
            except KeyError:
                std_estimates.append(np.nan)
        elif row['op'] == '=~':  # 요인 부하량
            try:
                x_std = np.std(data[row['rhs']], ddof=1)
                std_estimates.append(row['Estimate'] * x_std)
            except KeyError:
                std_estimates.append(np.nan)
        else:
            std_estimates.append(np.nan)
    df['std_estimate'] = std_estimates
    return df

# 학업압박

In [ ]:
df_edu = df[['q071','q072', 'q073', 'q074'] + ['q485', 'q486', 'q487', 'q488']]
df_edu = null0(df_edu)

df_edu = encoding(df_edu, ['q071', 'q072', 'q073', 'q074'], category4_map)
df_edu = encoding(df_edu, ['q485', 'q486','q487', 'q488'], category5_map)

In [ ]:
scaler = MinMaxScaler()
df_edu[['q071', 'q072', 'q073', 'q074', 'q485', 'q486','q487', 'q488']] = scaler.fit_transform(df_edu[['q071', 'q072', 'q073', 'q074', 'q485', 'q486','q487', 'q488']])

In [ ]:
model_desc = """
academic_stress =~ q072 + q073
exam_anxiety =~ q485 + q486 + q487 + q488
academic_pressure =~  academic_stress + exam_anxiety
academic_stress ~~ exam_anxiety
"""

In [ ]:
df_edu['wt'] = df['wt']
observed_vars = ['q072', 'q073', 'q485', 'q486', 'q487', 'q488']
w_cov_df = calculate_covariance(df_edu, observed_vars)

In [ ]:
model = Model(model_desc)
model = fit_sem_model(model, df_edu, w_cov_df)
stats = calc_stats(model)
estimates = model.inspect()
print("적합도 지표:\n", stats.T, "\n\n")
print("모수 추정치:\n", estimates)


semplot(model, 'C:\\git_files\\education_analysis\\happy\\academic_pressure.png')

In [ ]:
factor_scores = model.predict_factors(df_edu)

df["academic_pressure"] = factor_scores["academic_pressure"]
df['academic_stress'] = factor_scores["academic_stress"]
df['exam_anxiety'] = factor_scores["exam_anxiety"]

print(df.head())

In [ ]:
plt.hist(df["academic_pressure"], bins=20, color='blue')
plt.title("Distribution of academic pressure Factor")
plt.xlabel("academic presure Factor")
plt.ylabel("Frequency")
plt.show()

# 인간 관계

In [ ]:
df_rel = df[['q05','q134', 'q136', 'q489','q061','q062','q161','q162','q163','q164','q171','q172','q173','q174','q224','q225', 'q132']]
df_rel.columns

In [ ]:
df_rel['q172'].value_counts()

In [ ]:
df_rel['q05'].value_counts()

In [ ]:
df_rel = not_applicable(df_rel, ['q171','q172','q162', 'q161', 'q163', 'q164', 'q173', 'q174', 'q224'])
df_rel = applicable_is_the_best(df_rel, ['q132', 'q134'])
df_rel = applicable_is_the_worst(df_rel, ['q136'])
df_rel = encoding(df_rel, ['q05', 'q136','q061','q062','q161','q162','q163','q164','q171','q172','q173','q174','q224','q225'], category5_map)
df_rel = encoding(df_rel, ['q132', 'q134', 'q489'], reverse_category5_map)
df_rel = pd.DataFrame(scaler.fit_transform(df_rel), columns=df_rel.columns)
df_rel.head(3)

In [ ]:
model_desc = """
friend_rel =~ q132 + q134 + q136 + q489 + q05
teacher_rel =~ q061 + q062
father_rel =~ q161 + q163
mother_rel =~ q171 + q173
parent_rel =~ father_rel + mother_rel
group_rel =~ q224 + q225

human_rel =~ friend_rel + teacher_rel + parent_rel + group_rel

father_rel ~~ mother_rel
"""

In [ ]:
df_rel['wt'] = df['wt']
observed_vars = ['q05','q132','q134', 'q136','q489', 'q061', 'q062','q161','q163','q171','q173','q224', 'q225']
w_cov_df = calculate_covariance(df_rel, observed_vars)

In [ ]:
vif_df = compute_vif(df_rel, observed_vars)
model = Model(model_desc)
model = fit_sem_model(model, df_rel, w_cov_df)
stats = calc_stats(model)
estimates = model.inspect()
print("적합도 지표:\n", stats.T, "\n\n")
print("모수 추정치:\n", estimates)

semplot(model, 'C:\\git_files\\education_analysis\\happy\\human_rel.png')

In [ ]:
# 문제된 변수만 추출
rel_corr_df = df_rel[observed_vars].copy()

# 상관행렬 계산
corr_matrix = rel_corr_df.corr()

# 상관계수 히트맵 시각화
plt.figure(figsize=(6, 4))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation matrix of emotional state variables')
plt.show()


In [ ]:
factor_scores = model.predict_factors(df_rel)

df["human_rel"] = factor_scores["human_rel"]
df['teacher_rel'] = factor_scores["teacher_rel"]
df['friend_rel'] = factor_scores["friend_rel"]
df['parent_rel'] = factor_scores["parent_rel"]
df['group_rel'] = factor_scores["group_rel"]

print(df.head())

In [ ]:
plt.hist(df["human_rel"], bins=20, color='blue')
plt.title("Distribution of Human Relationship Factor")
plt.xlabel("Human Relationship Factor")
plt.ylabel("Frequency")
plt.show()

# 정서적 상태

In [ ]:
emo = df[['q221', 'q222', 'q223', 'q226', 'q23', 'q26']]
emo.isnull().sum()

In [ ]:
for i in emo.columns:
    display(emo[i].value_counts())

In [ ]:
emo = encoding(emo, ['q221', 'q222', 'q223'], category5_map)
emo = encoding(emo, ['q226'], reverse_category5_map)
emo = encoding(emo, ['q23'], happy_map)
emo = encoding(emo, ['q26'], how_happy_map)

In [ ]:
emo.info()

In [ ]:
scaler = MinMaxScaler()
emo = pd.DataFrame(scaler.fit_transform(emo), columns=emo.columns)

In [ ]:
emo.info()

In [ ]:
model_desc = """
emotional_state =~ q221 + q223 + q226 + q23
"""

In [ ]:
emo['wt'] = df['wt']
observed_vars = ['q221', 'q223', 'q226', 'q23']
w_cov_df = calculate_covariance(emo, observed_vars)

In [ ]:
vif_df = compute_vif(emo, observed_vars)
model = Model(model_desc)
model = fit_sem_model(model, emo, w_cov_df)
stats = calc_stats(model)
estimates = model.inspect()
print("적합도 지표:\n", stats.T, "\n\n")
print("모수 추정치:\n", estimates)


semplot(model, 'C:\\git_files\\education_analysis\\happy\\emotional_state.png')

In [ ]:
factor_scores = model.predict_factors(emo)

df["emotional_state"] = factor_scores["emotional_state"]

print(df.head())

In [ ]:
plt.hist(df["emotional_state"], bins=20, color='blue')
plt.title("Distribution of Emotional State Factor")
plt.xlabel("Emotional State Factor")
plt.ylabel("Frequency")
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 문제된 변수만 추출
emo_vars = ['q221', 'q223', 'q226', 'q23']
emo_corr_df = emo[emo_vars].copy()

# 상관행렬 계산
corr_matrix = emo_corr_df.corr()

# 상관계수 히트맵 시각화
plt.figure(figsize=(6, 4))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation matrix of emotional state variables')
plt.show()


In [ ]:
print(df[df['q35'] == '있다'])

# 학교 폭력 노출도

In [ ]:
df_vio = df[['q33', 'q35', 'q351', 'q352']]

In [ ]:
display(df_vio['q33'].value_counts())
display(df_vio['q35'].value_counts())
display(df_vio['q351'].value_counts())
display(df_vio['q352'].value_counts())

In [ ]:
df_vio.info()

In [ ]:
df_vio = encoding(df_vio, ['q33'], how_serious_map)
df_vio = encoding(df_vio, ['q35'], is_or_not_map)
df_vio['q351'] = df_vio['q351'].replace({'모름/무응답' : 2})
df_vio['q351'] = df_vio['q351'].apply(lambda x: 2 if x >= 2 else (1 if x == 1 else 0))
df_vio['q352'] = df_vio['q352'].replace({'모름/무응답' : 2})
df_vio['q352'] = df_vio['q352'].apply(lambda x: 2 if x >= 2 else (1 if x == 1 else 0))
df_vio = null0(df_vio)
df_vio

In [ ]:
display(df_vio['q351'].value_counts())
display(df_vio['q352'].value_counts())

In [ ]:
model_desc = """
exposure_violence =~ q33 + q35 + q351 + q352
"""

In [ ]:

df_vio['wt'] = df['wt']
observed_vars = ['q33', 'q35', 'q351', 'q352']
w_cov_df = calculate_covariance(df_vio, observed_vars)
model = Model(model_desc)
model = fit_sem_model(model, df_vio, w_cov_df)
stats = calc_stats(model)
estimates = model.inspect()
print("적합도 지표:\n", stats.T, "\n\n")
print("모수 추정치:\n", estimates)

In [ ]:
semplot(model, 'C:\\git_files\\education_analysis\\happy\\school_violence.png')


In [ ]:
factor_scores = model.predict_factors(df_vio)

df["exposure_violence"] = factor_scores["exposure_violence"]

print(df.head())
plt.hist(df["exposure_violence"], bins=20, color='blue')
plt.title("Distribution of exposure violence Factor")
plt.xlabel("exposure violence Factor")
plt.ylabel("Frequency")
plt.show()

In [ ]:
df['exposure_violence'].value_counts()

# 외모 및 신체 이미지

In [ ]:
body = df[['q42', 'q43', 'q4812', 'q4813', 'q4814', 'q4815']]
body.isnull().sum()

In [ ]:
for i in body.columns:
    display(body[i].value_counts())

In [ ]:
body = encoding(body, ['q4812', 'q4813', 'q4814', 'q4815'], category5_map)
body = calculate_bmi(body, 'q42', 'q43')
body['obesity'] = body['bmi'].apply(lambda x: (
    0 if x < 18.5 else
    1 if x < 23 else
    2 if x < 25 else
    3 if x < 30 else
    4 if x < 35 else
    5
))

body['obesity'] = body['obesity'].astype('category')

In [ ]:
scaler = MinMaxScaler()
body[['q42', 'q43']] = scaler.fit_transform(body[['q42', 'q43']])

In [ ]:
body.info()

In [ ]:
body.head(3)

In [ ]:
model_desc = '''
body_stress =~ q4812 + q4813 + q4814 + q4815
body_info =~ obesity

body_image_experience =~ body_stress + body_info

q4812 ~~ q4813
q4812 ~~ obesity
'''

In [ ]:
body['wt'] = df['wt']
observed_vars = ['q4812', 'q4813', 'q4814','q4815', 'obesity', 'q42', 'q43']
w_cov_df = calculate_covariance(body, observed_vars)
vif_df = compute_vif(body, observed_vars)
model = Model(model_desc)
model = fit_sem_model(model, body, w_cov_df)
stats = calc_stats(model)
estimates = model.inspect()
print("적합도 지표:\n", stats.T, "\n\n")
print("모수 추정치:\n", estimates)

In [ ]:
semplot(model, 'C:\\git_files\\education_analysis\\happy\\body_image_experience.png')

In [ ]:
factor_scores = model.predict_factors(body)

df["body_image_experience"] = factor_scores["body_image_experience"]

print(df.head())


In [ ]:
plt.hist(df["body_image_experience"], bins=20, color='blue')
plt.title("Distribution of body image experience")
plt.xlabel("body experience")
plt.ylabel("Frequency")
plt.show()
df['body_image_experience'].value_counts()

# 부모의 경제적 지원

In [ ]:
# 재정적 지원
financial_support = ['q38', 'q4816', 'q4817']
time_support = ['q162', 'q172']
support = pd.concat([df[financial_support], df[time_support]], axis=1)
support['need_money'] = df['q27'].apply(lambda x: 0 if x == '돈' else 1)
support.head(3)

In [ ]:
for i in support.columns:
    display(support[i].value_counts())

In [ ]:
support = encoding(support, ['q38'], how_high_map)
support = not_applicable(support, ['q4816', 'q162','q172'])
support = encoding(support, ['q4816', 'q4817'], reverse_category5_map)
support = encoding(support, ['q162', 'q172'], category5_map)
support['need_money'].astype('category')
support.head(3)

In [ ]:
scaler = MinMaxScaler()
support[['q4816', 'q4817', 'q162', 'q172']] = scaler.fit_transform(support[['q4816', 'q4817', 'q162', 'q172']])

In [ ]:
support['q38'] = pd.Categorical(support['q38'], categories=[0, 1, 2], ordered=True)

In [ ]:
support.info()

In [ ]:
model_desc = """
    financial_support =~ q4816 + q4817
    financial_environment =~ q38 + need_money
    time_support =~ q162 + q172
    support =~ financial_support + time_support
    financial_support ~~ financial_environment
    q4816 ~~ q4817
    financial_support ~~ financial_environment
"""


In [ ]:
support['wt'] = df['wt']
observed_vars = ['q38', 'q4816', 'q4817', 'q162', 'q172', 'need_money']
w_cov_df = calculate_covariance(support, observed_vars)

In [ ]:
model = Model(model_desc)
model = fit_sem_model(model, support, w_cov_df)
stats = calc_stats(model)
estimates = model.inspect()
print("적합도 지표:\n", stats.T, "\n\n")
print("모수 추정치:\n", estimates)


semplot(model, 'C:\\git_files\\education_analysis\\happy\\support.png')

In [ ]:
factor_scores = model.predict_factors(support)

df["support"] = factor_scores["support"]
df['time_support'] = factor_scores["time_support"]
df['financial_support'] = factor_scores["financial_support"]
df['financial_environment'] = factor_scores["financial_environment"]
print(df.head())

plt.hist(df["support"], bins=20, color='blue')
plt.title("Distribution of support")
plt.xlabel("support")
plt.ylabel("Frequency")
plt.show()
df['support'].value_counts()

# 위험행동

In [ ]:
df['q302'].value_counts()

In [ ]:
crisis = df[['q287', 'q288', 'q289', 'q2810','q29', 'q31']]
crisis.isnull().sum()

In [ ]:
crisis.info()

In [ ]:
crisis = encoding(crisis, ['q287', 'q288', 'q289', 'q2810'], yes_or_no_map)
crisis = encoding(crisis, ['q29', 'q31'], how_much_is_map)
# crisis = encoding(crisis, ['q302'], is_or_not_map)

crisis = null0(crisis)

In [ ]:
model_desc = """
    substance =~  q287 + q288 + q289
    impulse =~ q29 + q31
    crisis_behavior =~ substance + q2810 + impulse
"""

In [ ]:
crisis['wt'] = df['wt']
observed_vars = ['q287', 'q288', 'q289', 'q2810', 'q29', 'q31']
w_cov_df = calculate_covariance(crisis, observed_vars)

In [ ]:
model = Model(model_desc)
model = fit_sem_model(model, crisis, w_cov_df)
stats = calc_stats(model)
estimates = model.inspect()
print("적합도 지표:\n", stats.T, "\n\n")
print("모수 추정치:\n", estimates)


semplot(model, 'C:\\git_files\\education_analysis\\happy\\crisis.png')

In [ ]:
factor_scores = model.predict_factors(crisis)

df["crisis_behavior"] = factor_scores["crisis_behavior"]
df['impulse'] = factor_scores["impulse"]
df['substance'] = factor_scores["substance"]

print(df.head())

plt.hist(df["crisis_behavior"], bins=20, color='blue')
plt.title("Distribution of crisis_behavior")
plt.xlabel("crisis_behavior")
plt.ylabel("Frequency")
plt.show()
df['crisis_behavior'].value_counts()

# 조절효과 회귀분석 (OLS)

In [ ]:
# df = df[df['school'] == '고등학생']
# df.head(3)

### emotional_state ~ academic_pressure + rel

In [ ]:
df['interaction'] = df['academic_pressure'] * df['group_rel']

model = smf.ols('emotional_state ~ academic_pressure + group_rel + interaction', data=df).fit()
print(model.summary())

In [ ]:
df['interaction'] = df['academic_pressure'] * df['teacher_rel']

model = smf.ols('emotional_state ~ academic_pressure + teacher_rel + interaction', data=df).fit()
print(model.summary())

In [ ]:
df["interaction"] = df["academic_pressure"] * df["parent_rel"]

model = smf.ols("emotional_state ~ academic_pressure + parent_rel + interaction", data=df).fit()
print(model.summary())

In [ ]:
df['interaction'] = df['academic_pressure'] * df['friend_rel']

model = smf.ols('emotional_state ~ academic_pressure + friend_rel + interaction', data=df).fit()
print(model.summary())

### impulse ~ violence + rel

In [ ]:
df['interaction'] = df['exposure_violence'] * df['group_rel']

model = smf.ols('impulse ~ exposure_violence + group_rel + interaction', data=df).fit()
print(model.summary())

In [ ]:
df['interaction'] = df['exposure_violence'] * df['teacher_rel']

model = smf.ols('impulse ~ exposure_violence + teacher_rel + interaction', data=df).fit()
print(model.summary())

In [ ]:
df['interaction'] = df['exposure_violence'] * df['parent_rel']

model = smf.ols('impulse ~ exposure_violence + parent_rel + interaction', data=df).fit()
print(model.summary())

In [ ]:
df['interaction'] = df['exposure_violence'] * df['friend_rel']

model = smf.ols('impulse ~ exposure_violence + friend_rel + interaction', data=df).fit()
print(model.summary())

### impulse ~ emotional_state + rel

In [ ]:
df['interaction'] = df['emotional_state'] * df['group_rel']

model = smf.ols('impulse ~ emotional_state + group_rel + interaction', data=df).fit()
print(model.summary())

In [ ]:
df['interaction'] = df['emotional_state'] * df['teacher_rel']

model = smf.ols('impulse ~ emotional_state + teacher_rel + interaction', data=df).fit()
print(model.summary())

In [ ]:
df['interaction'] = df['emotional_state'] * df['parent_rel']

model = smf.ols('impulse ~ emotional_state + parent_rel + interaction', data=df).fit()
print(model.summary())

In [ ]:
df['interaction'] = df['emotional_state'] * df['friend_rel']

model = smf.ols('impulse ~ emotional_state + friend_rel + interaction', data=df).fit()
print(model.summary())

In [ ]:
df['interaction'] = df['emotional_state'] * df['human_rel']

model = smf.ols('impulse ~ emotional_state + human_rel + interaction', data=df).fit()
print(model.summary())

### Y ~ X + support

In [ ]:
df['interaction'] = df['academic_pressure'] * df['support']

model = smf.ols('emotional_state ~ academic_pressure + support + interaction', data=df).fit()
print(model.summary())

In [ ]:
df['interaction'] = df['academic_pressure'] * df['support']

model = smf.ols('impulse ~ academic_pressure + support + interaction', data=df).fit()
print(model.summary())

In [ ]:
df['interaction'] = df['academic_pressure'] * df['support']

model = smf.ols('parent_rel ~ academic_pressure + support + interaction', data=df).fit()
print(model.summary())

In [ ]:
df['interaction'] = df['emotional_state'] * df['support']

model = smf.ols('parent_rel ~ emotional_state + support + interaction', data=df).fit()
print(model.summary())

In [ ]:
df['interaction'] = df['emotional_state'] * df['support']

model = smf.ols('impulse ~ emotional_state + support + interaction', data=df).fit()
print(model.summary())

In [ ]:
df['interaction'] = df['emotional_state'] * df['support']

model = smf.ols('crisis_behavior ~ emotional_state + support + interaction', data=df).fit()
print(model.summary())

In [ ]:
df['interaction'] = df['human_rel'] * df['support']

model = smf.ols('academic_pressure ~ human_rel + support + interaction', data=df).fit()
print(model.summary())

In [ ]:
df['interaction'] = df['financial_environment'] * df['support']

model = smf.ols('emotional_state ~ financial_environment + support + interaction', data=df).fit()
print(model.summary())

In [ ]:
df['interaction'] = df['academic_stress'] * df['support']

model = smf.ols('crisis_behavior ~ academic_stress + support + interaction', data=df).fit()
print(model.summary())

# 종합

In [ ]:
everything = pd.concat([df_edu, df_rel, emo, body, df_vio, crisis, support], axis=1)

In [ ]:
everything = everything.loc[:, ~everything.columns.duplicated()]

In [ ]:
everything.head(3)

In [ ]:
model_desc = """
academic_stress =~ q072 + q073
exam_anxiety =~ q485 + q486 + q487 + q488
academic_pressure =~  academic_stress + exam_anxiety

friend_rel =~ q132 + q134 + q136 + q489 + q05
teacher_rel =~ q061 + q062
father_rel =~ q161 + q163
mother_rel =~ q171 + q173
parent_rel =~ father_rel + mother_rel
group_rel =~ q224 + q225
human_rel =~ friend_rel + teacher_rel + parent_rel + group_rel
father_rel ~~ mother_rel

emotional_state =~ q221 + q223 + q226 + q23

body_image_stress =~ q4812 + q4813 + q4814 + q4815
body_info =~ obesity

body_image_experience =~ body_image_stress
body_image_experience ~~ body_info

exposure_violence =~ q35 + q351 + q352

substance =~  q287 + q288 + q289
impulse =~ q29 + q31
crisis_behavior =~ substance + q2810 + impulse

financial_support =~ q4816 + q4817 + need_money
financial_environment =~ q38
support =~ financial_support
financial_support ~~ financial_environment

# [1] 정서 상태(emotional_state)에 영향을 주는 요인들
emotional_state ~ academic_pressure          # 학업 압박이 정서 상태에 영향

emotional_state ~ human_rel                  # 인간 관계 → 정서 안정
emotional_state ~ parent_rel                 # 부모와의 관계 → 정서 상태
emotional_state ~ friend_rel                 # 친구 관계 → 정서 상태

emotional_state ~ body_image_experience      # 신체 이미지 경험 → 정서 상태
emotional_state ~ academic_stress            # 학업 스트레스 → 정서 상태
emotional_state ~ exposure_violence          # 폭력 노출 → 정서 상태
emotional_state ~ support

# [2] 충동성과 위기행동에 영향을 주는 정서 상태
impulse ~ emotional_state                    # 감정 상태 → 충동성
crisis_behavior ~ emotional_state            # 정서 상태 → 위기 행동
substance ~ emotional_state

# [3] 폭력 노출의 영향
impulse ~ exposure_violence                  # 폭력 경험 → 충동성
human_rel ~ exposure_violence                # 폭력 경험 → 인간관계 악화
substance ~ exposure_violence

# [5] 학업 스트레스 관련 경로
academic_pressure ~ human_rel
academic_pressure ~ teacher_rel
academic_pressure ~  parent_rel
academic_pressure ~ support
academic_pressure ~ emotional_state

academic_stress ~ human_rel                  # 좋은 인간 관계 → 학업 스트레스 완충
academic_stress ~ teacher_rel             # 시험 불안 → 학업 압박
academic_stress ~ parent_rel                 # 부모 관계 → 학업 스트레스
academic_stress ~ support
academic_stress ~ emotional_state

exam_anxiety ~ human_rel
exam_anxiety ~ teacher_rel
exam_anxiety ~  parent_rel
exam_anxiety ~ support
exam_anxiety ~ emotional_state

# [6] 인간 관계에 영향을 주는 경로
human_rel ~ academic_pressure                # 학업 압박 → 인간 관계 악화
human_rel ~ substance
human_rel ~ support
parent_rel ~ substance                       # 물질 사용 → 부모 관계 악화
parent_rel ~ academic_stress
parent_rel ~ academic_pressure
parent_rel ~ exam_anxiety
parent_rel ~ financial_environment          # 부모 관계 → 경제 환경
friend_rel ~ exposure_violence
teacher_rel ~ substance
teacher_rel ~ exposure_violence                # 폭력 노출 → 교사 관계 악화
teacher_rel ~ exam_anxiety
teacher_rel ~ academic_pressure

# [7] 충동성과 위기행동
impulse ~ academic_pressure                  # 학업 압박 → 충동성
impulse ~ friend_rel                         # 친구 관계 → 충동 억제
impulse ~ parent_rel
impulse ~ exam_anxiety
impulse ~ exposure_violence                 # 폭력 노출 → 충동성
impulse ~ support
impulse ~ human_rel

substance ~ academic_pressure      
substance ~ friend_rel
substance ~ parent_rel
substance ~ exam_anxiety
substance ~ exposure_violence      
substance ~ support            
substance ~ human_rel

crisis_behavior ~ academic_pressure          # 학업 압박 → 위기 행동
crisis_behavior ~ friend_rel
crisis_behavior ~ parent_rel
crisis_behavior ~ exam_anxiety
crisis_behavior ~ exposure_violence          # 폭력 노출 → 위기 행동
crisis_behavior ~ support                   # 가족 지원 → 위기 행동 완화
crisis_behavior ~ human_rel

# [8] 가족의 지원(support)의 영향
emotional_state ~ support                   # 가족 지원 → 정서 안정
human_rel ~ support                         # 가족 지원 → 인간관계 향상
academic_stress ~ support                   # 가족 지원 → 학업 스트레스 완충
academic_pressure ~ support                 # 가족 지원 → 학업 압박 완충
exam_anxiety ~ support                      # 가족 지원 → 시험 불안 완화
impulse ~ support                           # 가족 지원 → 충동성 억제
crisis_behavior ~ support                   # 가족 지원 → 위기 행동 완화
emotional_state ~ financial_environment     # 경제 환경 → 정서 안정
parent_rel ~ support                        # 부모 관계 → 가족 지원
parent_rel ~ financial_environment          # 부모 관계 → 경제 환경
"""

observed_vars = [
    'q072', 'q073', 'q485', 'q486', 'q487', 'q488', 'q221', 'q223', 'q226', 'q23',
    'q05','q132', 'q134', 'q136', 'q489', 'q061', 'q062', 'q161', 'q163', 'q171', 'q173', 'q224', 'q225',
    'q4812', 'q4813', 'q4814', 'q4815', 'obesity', 'q42', 'q43',
    'q33', 'q35', 'q351', 'q352', 
    'q287', 'q288', 'q289', 'q2810','q29', 'q31',
    'q4816', 'q4817', 'q38', 'need_money', 'q162', 'q172'
                ]
w_cov_df = calculate_covariance(everything, observed_vars)
vif_df = compute_vif(everything, observed_vars)
model = Model(model_desc)
model = fit_sem_model(model, everything, w_cov_df)
stats = calc_stats(model)
estimates = model.inspect()
print("적합도 지표:\n", stats.T, "\n\n")
print("모수 추정치:\n", estimates)

semplot(model, 'C:\\git_files\\education_analysis\\happy\\everything.png')
report(model, 'C:\\git_files\\education_analysis\\happy\\report')

In [ ]:
# 각 하위 요인별 변수 목록
academic_stress_columns = ['q072', 'q073']
exam_anxiety_columns = ['q485', 'q486', 'q487', 'q488']
friend_rel_columns = ['q132', 'q134', 'q136', 'q489', 'q05']
teacher_rel_columns = ['q061', 'q062']
father_rel_columns = ['q161', 'q163']
mother_rel_columns = ['q171', 'q173']
parent_rel_columns = ['q161', 'q163', 'q171', 'q173']
group_rel_columns = ['q224', 'q225']
emotional_state_columns = ['q221', 'q223', 'q226', 'q23']
body_image_stress_columns = ['q4812', 'q4813', 'q4814', 'q4815']
exposure_violence_columns = ['q35', 'q351', 'q352']
substance_columns = ['q287', 'q288', 'q289']
impulse_columns = ['q29', 'q31']
financial_support_columns = ['q4816', 'q4817', 'need_money']

for col in academic_stress_columns + exam_anxiety_columns + friend_rel_columns + teacher_rel_columns + \
        father_rel_columns + mother_rel_columns + parent_rel_columns + group_rel_columns + \
        emotional_state_columns + body_image_stress_columns + \
        exposure_violence_columns + substance_columns + impulse_columns + \
        financial_support_columns:
    # 각 열을 수치형으로 변환
    everything[col] = pd.to_numeric(everything[col], errors='coerce')


# 각 하위 요인별 Cronbach's α 계산 함수
def calculate_cronbach_alpha(columns):
    data = everything[columns]
    return pg.cronbach_alpha(data=data)[0]

# 각 하위 요인별 Cronbach's α 계산
cronbach_results = {}

# 각 하위 요인에 대해 Cronbach's α 계산
cronbach_results['academic_stress'] = calculate_cronbach_alpha(academic_stress_columns)
cronbach_results['exam_anxiety'] = calculate_cronbach_alpha(exam_anxiety_columns)
cronbach_results['friend_rel'] = calculate_cronbach_alpha(friend_rel_columns)
cronbach_results['teacher_rel'] = calculate_cronbach_alpha(teacher_rel_columns)
cronbach_results['father_rel'] = calculate_cronbach_alpha(father_rel_columns)
cronbach_results['mother_rel'] = calculate_cronbach_alpha(mother_rel_columns)
cronbach_results['parent_rel'] = calculate_cronbach_alpha(parent_rel_columns)
cronbach_results['group_rel'] = calculate_cronbach_alpha(group_rel_columns)
cronbach_results['emotional_state'] = calculate_cronbach_alpha(emotional_state_columns)
cronbach_results['body_image_stress'] = calculate_cronbach_alpha(body_image_stress_columns)
cronbach_results['exposure_violence'] = calculate_cronbach_alpha(exposure_violence_columns)
cronbach_results['substance'] = calculate_cronbach_alpha(substance_columns)
cronbach_results['impulse'] = calculate_cronbach_alpha(impulse_columns)
cronbach_results['financial_support'] = calculate_cronbach_alpha(financial_support_columns)

# Cronbach's α 값 출력
for factor, alpha in cronbach_results.items():
    print(f"{factor}의 Cronbach's α: {alpha:.3f}")


In [ ]:
model = Model(model_desc)
model.fit(everything)

# 3. 추정 결과 추출
estimates = model.inspect()
estimates['p-value'] = pd.to_numeric(estimates['p-value'], errors='coerce')
significant_paths = estimates[(estimates['p-value'] < 0.05) & (estimates['op'] == '~')]

# 4. 유효한 식만 추출
valid_equations = []

for _, row in significant_paths.iterrows():
    lhs = row['lval']
    rhs = row['rval']
    op = row['op']
    Estimate = row['Estimate']
    valid_equations.append(f"{lhs} | {op} | {rhs} | {Estimate}")

# 중복 제거 + 정렬
valid_equations = sorted(set(valid_equations))

# 5. 최종 모델식 만들기
model_desc_valid = "\n".join(valid_equations)

# 6. 확인
print("✅ 유효한 가설만 남은 모델식:\n")
print(model_desc_valid)


In [ ]:
model = Model(model_desc)
model.fit(everything)

# 3. 추정 결과 추출
estimates = model.inspect()
estimates['p-value'] = pd.to_numeric(estimates['p-value'], errors='coerce')
significant_paths = estimates[(estimates['p-value'] > 0.05) & (estimates['p-value'] <= 0.1) & (estimates['op'] == '~')]

# 4. 유효한 식만 추출
valid_equations = []

for _, row in significant_paths.iterrows():
    lhs = row['lval']
    rhs = row['rval']
    op = row['op']
    Estimate = row['Estimate']
    pvalue = row['p-value']
    valid_equations.append(f"{lhs} | {op} | {rhs} | {Estimate} | {pvalue}")

# 중복 제거 + 정렬
valid_equations = sorted(set(valid_equations))

# 5. 최종 모델식 만들기
model_desc_valid = "\n".join(valid_equations)

# 6. 확인
print("✅ 유효한 가설만 남은 모델식:\n")
print(model_desc_valid)


In [ ]:
def format_value(x):
    try:
        if str(x).strip() == '-':
            return x
        return f"{float(x):.6f}"
    except:
        return x

In [ ]:
def standardize_estimates_full(estimates_df, data):
    """
    semopy 모델의 결과 DataFrame(estimates_df)에 대해 표준화 계수를 계산하는 함수.
    - 회귀 계수(op='~') 및 요인 부하량(op='=~')에 대해 계산.
    
    Parameters:
        estimates_df (pd.DataFrame): semopy의 model.inspect() 결과
        data (pd.DataFrame): 원자료 또는 잠재변수 점수가 포함된 데이터프레임

    Returns:
        pd.DataFrame: std_estimate 열이 추가된 DataFrame
    """
    std_estimates = []

    for _, row in estimates_df.iterrows():
        op = row['op']
        lhs = row['lval']
        rhs = row['rval']
        estimate = row['Estimate']

        if op == '~':  # 회귀 관계: lhs ~ rhs
            if lhs in data.columns and rhs in data.columns:
                try:
                    std_lhs = np.std(data[lhs], ddof=1)
                    std_rhs = np.std(data[rhs], ddof=1)
                    std_beta = estimate * (std_rhs / std_lhs)
                    std_estimates.append(std_beta)
                except Exception:
                    std_estimates.append(np.nan)
            else:
                std_estimates.append(np.nan)

        elif op == '=~':  # 요인 부하량: latent =~ indicator
            if rhs in data.columns:
                try:
                    std_rhs = np.std(data[rhs], ddof=1)
                    std_loading = estimate * std_rhs  # 잠재변수는 단위분산 가정
                    std_estimates.append(std_loading)
                except Exception:
                    std_estimates.append(np.nan)
            else:
                std_estimates.append(np.nan)

        else:
            std_estimates.append(np.nan)

    estimates_df['std_estimate'] = std_estimates
    return estimates_df


In [ ]:
factor_scores = model.predict_factors(everything)
combined_df = pd.concat([everything.copy(), factor_scores], axis=1)
estimates_df = model.inspect()
std_estimates_df = standardize_estimates_full(estimates_df, combined_df)

# 결과 확인
pd.set_option('display.max_rows', None)
display(std_estimates_df)
pd.reset_option('display.max_rows')

In [ ]:
for col in ['Estimate', 'Std. Err', 'z-value', 'p-value', 'std_estimate']:
    if col in std_estimates_df.columns:
        std_estimates_df[col] = std_estimates_df[col].apply(format_value)

std_estimates_df.to_csv('c:\\git_files\\education_analysis\\happy\\estimates.csv', index=False)
df.to_csv('c:\\git_files\\education_analysis\\happy\\final_df.csv', index=False)